# StreamDiffusion Remote TouchDesigner Bridge
Run SD-Turbo on a Colab GPU and stream output to TouchDesigner over a Cloudflare tunnel.

1. Select a GPU runtime, then run the four code cells in order.
2. The launcher checks actual server readiness before creating the tunnel.
3. Paste the displayed `wss://.../ws/stream` URL into TouchDesigner and enable Connect.

Supports text/image generation, prompt blending, audio controls, telemetry, snapshots,
and 16 output presets up to 1024 x 768 (or 768 x 1024 portrait). Default: 512 x 512.
Resolution changes reload model buffers; TensorRT engines are cached per width/height.
TensorRT uses matching Torch-TensorRT wheels on Python 3.11+ / PyTorch 2.11+.
Both modes compile and validate their UNets; TinyVAE stays in PyTorch. Requested
acceleration never silently falls back to a successful native startup.

Setup pins the StreamDiffusion source revision and core compatibility dependencies.
Re-run the launch cell to replace its own previous server and tunnel. Interrupt it to stop both.
Server and tunnel logs are saved in `bridge_logs/` inside the Colab runtime.
[Project source](https://github.com/wenjunii/colab-sd)


In [ ]:
# Step 1: Check GPU
import subprocess
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then run this cell again.')
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# Step 2: Install the compatible environment
"""Colab setup source embedded into the generated notebook."""
import os
from pathlib import Path
import platform
import subprocess
import sys
import urllib.request

UPSTREAM_REVISION = 'b623251dc055e1fd858d53509aa43e09dfc5cdc0'
CORE_DEPENDENCIES = (
    'diffusers==0.24.0', 'transformers==4.39.3',
    'huggingface-hub==0.25.2', 'accelerate==0.30.1',
    'fastapi', 'uvicorn', 'websockets', 'python-multipart',
    'opencv-python-headless', 'pillow',
)


def setup_environment(root='/content/StreamDiffusion'):
    if platform.system() != 'Linux':
        raise RuntimeError('This setup targets Google Colab Linux GPU runtimes.')
    root = Path(root).resolve()
    if not root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/cumulo-autumn/StreamDiffusion.git', str(root)], check=True)
    subprocess.run(['git', '-C', str(root), 'fetch', '--depth', '1', 'origin', UPSTREAM_REVISION], check=True)
    subprocess.run(['git', '-C', str(root), 'checkout', '--detach', UPSTREAM_REVISION], check=True)
    os.chdir(root)
    # Resolve the editable package and compatibility pins together so a later
    # install cannot silently undo earlier transformer/hub constraints.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', *CORE_DEPENDENCIES], check=True)
    target = root / 'cloudflared'
    if not target.exists():
        temporary = root / 'cloudflared.download'
        try:
            urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', temporary)
            temporary.replace(target)
        finally:
            temporary.unlink(missing_ok=True)
    target.chmod(0o755)
    print('Setup complete. StreamDiffusion revision:', UPSTREAM_REVISION)
    print('Run the next cell to write the bridge server and helpers.')
    return root


if __name__ == '__main__':
    setup_environment()


In [ ]:
# Step 3: Write server and standalone helpers
server_code = "import os\nimport sys\nimport traceback\nimport gc\n\n# 1. Compatibility shims for huggingface_hub\nimport huggingface_hub\nimport huggingface_hub.constants\n\nif not hasattr(huggingface_hub.constants, 'hf_cache_home'):\n    huggingface_hub.constants.hf_cache_home = getattr(\n        huggingface_hub.constants, 'HF_HOME', os.path.expanduser('~/.cache/huggingface')\n    )\n\nif not hasattr(huggingface_hub, 'HfFolder'):\n    class HfFolder:\n        @classmethod\n        def get_token(cls):\n            try:\n                from huggingface_hub import get_token\n                return get_token()\n            except Exception:\n                return None\n        @classmethod\n        def save_token(cls, token):\n            pass\n    huggingface_hub.HfFolder = HfFolder\n\nif not hasattr(huggingface_hub, 'cached_download'):\n    from huggingface_hub import hf_hub_download\n    huggingface_hub.cached_download = hf_hub_download\n\n# 2. Compatibility shims for transformers\nimport transformers\nimport transformers.utils\n\nif not hasattr(transformers.utils, 'FLAX_WEIGHTS_NAME'):\n    transformers.utils.FLAX_WEIGHTS_NAME = 'flax_model.msgpack'\nif not hasattr(transformers.utils, 'FLAX_WEIGHTS_INDEX_NAME'):\n    transformers.utils.FLAX_WEIGHTS_INDEX_NAME = 'flax_model.msgpack.index.json'\n\nif not hasattr(transformers, 'EncoderDecoderCache'):\n    class EncoderDecoderCache:\n        pass\n    transformers.EncoderDecoderCache = EncoderDecoderCache\n    try:\n        import transformers.cache_utils\n        transformers.cache_utils.EncoderDecoderCache = EncoderDecoderCache\n    except Exception:\n        pass\n\nimport time\nimport asyncio\nimport json\nimport logging\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor\nimport cv2\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom fastapi import FastAPI, WebSocket, WebSocketDisconnect\nfrom fastapi.responses import HTMLResponse\nfrom fastapi.responses import JSONResponse\nfrom tensorrt_backend import compile_unet as compile_tensorrt_unet\nfrom bridge_protocol import (OUTPUT_RESOLUTIONS, PROTOCOL_VERSION, OUTPUT_CHUNK_BYTES,\n                             MAX_INPUT_FRAME_BYTES, FrameAssembler, resolution, validate_controls)\n\n# Disable autograd globally for fast inference\ntorch.set_grad_enabled(False)\nif torch.cuda.is_available():\n    torch.backends.cuda.matmul.allow_tf32 = True\n    torch.backends.cudnn.allow_tf32 = True\n    torch.backends.cudnn.benchmark = True\n\nsys.path.append('.')\ntry:\n    from utils.wrapper import StreamDiffusionWrapper\n    from streamdiffusion.image_utils import postprocess_image\nexcept ImportError:\n    StreamDiffusionWrapper = None\n    postprocess_image = None\n\n# Patch image_utils.pt_to_numpy just in case\ntry:\n    import streamdiffusion.image_utils\n    orig_pt_to_numpy = streamdiffusion.image_utils.pt_to_numpy\n    def safe_pt_to_numpy(images):\n        return images.detach().cpu().permute(0, 2, 3, 1).float().numpy()\n    streamdiffusion.image_utils.pt_to_numpy = safe_pt_to_numpy\nexcept Exception:\n    pass\n\nlogging.basicConfig(level=logging.INFO)\nlogger = logging.getLogger(\"StreamDiffusionServer\")\napp = FastAPI()\n\nstream_txt2img = None\nstream_img2img = None\npipeline_ready = False\n# The two modes share model weights and mutable state. Keep all GPU work on\n# one worker, including control updates, without blocking WebSocket handling.\nmodel_executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix=\"diffusion\")\ntry:\n    max_stream_fps = float(os.environ.get(\"MAX_STREAM_FPS\", \"60\"))\n    if not np.isfinite(max_stream_fps) or not 1 <= max_stream_fps <= 120:\n        raise ValueError()\nexcept ValueError:\n    logger.warning(\"Invalid MAX_STREAM_FPS; using 60\")\n    max_stream_fps = 60.0\nprompt_embedding_cache = {}\napplied_prompt_state = None\ntelemetry_lock = threading.Lock()\ntelemetry_cache = (0.0, \"--\", \"--\")\nactive_client = None\npipeline_revision = 0\nlast_pipeline_error = None\n\n# Engine acceleration state\nenable_tensorrt = os.environ.get(\"ENABLE_TENSORRT\", \"0\") in (\"1\", \"true\", \"True\") or \"--tensorrt\" in sys.argv\nactive_engine = \"PyTorch (FP16)\"\nimg2img_engine = \"PyTorch (FP16)\"\n\n\ndef engine_for_mode(mode=None):\n    return img2img_engine if (mode or current_mode) == \"img2img\" else active_engine\n\n# Prompt state\ncurrent_prompt_a = \"cinematic cyberpunk cityscape, neon lights, 8k, detailed\"\ncurrent_prompt_b = \"bioluminescent alien jungle, glowing flora, ethereal, 8k\"\ncurrent_prompt_blend = 0.0\ncurrent_neg_prompt = \"blurry, low quality, distorted, watermark\"\n\ncurrent_mode = \"txt2img\"\ncurrent_seed = 42\ncurrent_strength = 0.6\noutput_width = 512\noutput_height = 512\noutput_resolutions = OUTPUT_RESOLUTIONS\n\n\ndef requested_resolution(data):\n    return resolution(data, (output_width, output_height))\n\ndef set_seed(s):\n    torch.manual_seed(s)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(s)\n\ndef apply_prompts_and_blend(prompt_a=None, prompt_b=None, blend=None, neg_prompt=None):\n    global current_prompt_a, current_prompt_b, current_prompt_blend, current_neg_prompt\n    global applied_prompt_state\n    if prompt_a is not None and str(prompt_a).strip():\n        current_prompt_a = str(prompt_a).strip()\n    if prompt_b is not None:\n        current_prompt_b = str(prompt_b).strip()\n    if blend is not None:\n        current_prompt_blend = max(0.0, min(1.0, float(blend)))\n    if neg_prompt is not None:\n        current_neg_prompt = str(neg_prompt).strip()\n\n    if not pipeline_ready or stream_txt2img is None:\n        return\n\n    state = (current_prompt_a, current_prompt_b, current_prompt_blend, current_neg_prompt)\n    if state == applied_prompt_state:\n        return\n\n    try:\n        device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        core_pipe = stream_txt2img.stream.pipe\n\n        def encode_cached(prompt):\n            key = (id(core_pipe), prompt, current_neg_prompt)\n            embedding = prompt_embedding_cache.pop(key, None)\n            if embedding is None:\n                embedding = core_pipe.encode_prompt(\n                    prompt=prompt,\n                    device=device,\n                    num_images_per_prompt=1,\n                    do_classifier_free_guidance=False,\n                    negative_prompt=current_neg_prompt,\n                )[0]\n                # Bound retained GPU memory while editing prompts interactively.\n                if len(prompt_embedding_cache) >= 4:\n                    del prompt_embedding_cache[next(iter(prompt_embedding_cache))]\n            prompt_embedding_cache[key] = embedding\n            return embedding\n\n        with torch.no_grad():\n            embed_a = encode_cached(current_prompt_a)\n\n            if current_prompt_b and current_prompt_blend > 0.001:\n                embed_b = encode_cached(current_prompt_b)\n                alpha = current_prompt_blend\n                blended = (1.0 - alpha) * embed_a + alpha * embed_b\n            else:\n                blended = embed_a\n\n            # Apply blended embedding tensor to both pipelines\n            for s in [stream_txt2img, stream_img2img]:\n                if s is not None:\n                    target = s.stream if hasattr(s, \"stream\") else s\n                    if hasattr(target, \"prompt_embeds\"):\n                        target.prompt_embeds = blended.repeat(target.batch_size, 1, 1)\n\n        applied_prompt_state = state\n        logger.debug(\"Prompt embeddings updated (blend=%.2f)\", current_prompt_blend)\n    except Exception as e:\n        logger.error(f\"Error in prompt blending: {traceback.format_exc()}\")\n        raise\n\ndef apply_strength(new_strength):\n    global current_strength, stream_img2img, applied_prompt_state\n    current_strength = max(0.1, min(1.0, float(new_strength)))\n    try:\n        if stream_img2img is not None:\n            if hasattr(stream_img2img, \"update_strength\"):\n                stream_img2img.update_strength(current_strength)\n            elif hasattr(stream_img2img, \"stream\") and hasattr(stream_img2img.stream, \"t_list\"):\n                t0 = max(5, min(44, int(45 - current_strength * 40)))\n                stream_img2img.stream.t_list = [t0, 45]\n                stream_img2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n                logger.info(f\"Updated img2img denoise strength to {current_strength:.2f} (t_index_list={[t0, 45]})\")\n            # prepare() replaces prompt_embeds; restore the cached blend afterward.\n            applied_prompt_state = None\n    except Exception as e:\n        logger.error(f\"Error updating strength: {e}\")\n        raise\n\ndef get_gpu_telemetry():\n    global telemetry_cache\n    # nvidia-smi can take tens of milliseconds. Sample at most once every 2s,\n    # and call this function off the asyncio event loop.\n    with telemetry_lock:\n        now = time.perf_counter()\n        if now - telemetry_cache[0] < 2.0:\n            return telemetry_cache[1:]\n        vram_str, temp_str = sample_gpu_telemetry()\n        telemetry_cache = (time.perf_counter(), vram_str, temp_str)\n        return vram_str, temp_str\n\n\ndef sample_gpu_telemetry():\n    vram_str = \"--\"\n    temp_str = \"--\"\n    if torch.cuda.is_available():\n        try:\n            allocated = torch.cuda.memory_allocated() / (1024**3)\n            total = torch.cuda.get_device_properties(0).total_memory / (1024**3)\n            pct = (allocated / total) * 100\n            vram_str = f\"{allocated:.1f}/{total:.1f}GB ({pct:.0f}%)\"\n        except Exception:\n            pass\n        try:\n            import subprocess\n            out = subprocess.check_output(\n                [\"nvidia-smi\", \"--query-gpu=temperature.gpu\", \"--format=csv,noheader,nounits\"],\n                timeout=1,\n                text=True\n            ).strip()\n            temp_str = f\"{out}\u00b0C\"\n        except Exception:\n            pass\n    return vram_str, temp_str\n\ndef run_model_inference(input_image=None, mode=None):\n    \"\"\"\n    Runs StreamDiffusion inference under torch.no_grad() inside the executor thread.\n    Never use torch.inference_mode() because StreamDiffusion caches tensors between steps.\n    \"\"\"\n    with torch.no_grad():\n        mode = mode or current_mode\n        if mode in (\"img2img\", \"v2v\") and input_image is not None and stream_img2img is not None:\n            return stream_img2img(input_image)\n        if stream_txt2img is not None:\n            return stream_txt2img()\n        return None\n\n\ndef process_frame(mode, frame_bytes=None):\n    \"\"\"Decode, infer, and encode on the worker; return complete frame timings.\"\"\"\n    started = time.perf_counter()\n    input_image = None\n    if frame_bytes is not None:\n        in_bgr = cv2.imdecode(np.frombuffer(frame_bytes, np.uint8), cv2.IMREAD_COLOR)\n        if in_bgr is None:\n            return None\n        input_image = Image.fromarray(cv2.cvtColor(in_bgr, cv2.COLOR_BGR2RGB)).resize((output_width, output_height))\n\n    inference_started = time.perf_counter()\n    output = run_model_inference(input_image, mode=mode)\n    inf_ms = (time.perf_counter() - inference_started) * 1000\n    if output is None:\n        return None\n    if isinstance(output, Image.Image):\n        out_bgr = cv2.cvtColor(np.asarray(output), cv2.COLOR_RGB2BGR)\n    elif torch.is_tensor(output):\n        arr = output.detach()\n        if arr.ndim == 4:\n            arr = arr[0]\n        arr = arr.cpu().permute(1, 2, 0).float().numpy()\n        out_bgr = cv2.cvtColor((np.clip(arr, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)\n    else:\n        out_bgr = np.asarray(output)\n    jpg_bytes = compress_frame(out_bgr)\n    return jpg_bytes, inf_ms, (time.perf_counter() - started) * 1000\n\n\ndef apply_controls(data):\n    \"\"\"Apply an entire update between inference calls on the same worker.\"\"\"\n    global current_mode, current_seed, output_width, output_height, last_pipeline_error, pipeline_ready\n    data = validate_controls(data, (output_width, output_height))\n    width, height = requested_resolution(data)\n    state_fields = ('current_mode', 'current_seed', 'current_strength', 'current_prompt_a',\n                    'current_prompt_b', 'current_prompt_blend', 'current_neg_prompt',\n                    'output_width', 'output_height')\n    previous = {name: globals()[name] for name in state_fields}\n    resizing = (width, height) != (output_width, output_height) or not pipeline_ready\n    try:\n        _apply_validated_controls(data, width, height, resizing)\n        last_pipeline_error = None\n    except Exception as exc:\n        globals().update(previous)\n        last_pipeline_error = \"Configuration failed; see Colab server logs\"\n        # A failed constructor can retain tensors in its traceback. Release\n        # those frames before attempting to allocate the old model buffers.\n        exc.__traceback__ = None\n        try:\n            if resizing:\n                load_pipeline()\n            else:\n                apply_strength(current_strength)\n                set_seed(current_seed)\n                apply_prompts_and_blend()\n        except Exception:\n            pipeline_ready = False\n            logger.exception(\"Could not restore the previous pipeline configuration\")\n        raise RuntimeError(f\"Configuration failed: {exc}\") from exc\n\n\ndef _apply_validated_controls(data, width, height, resizing):\n    global current_mode, current_seed, output_width, output_height\n    global current_prompt_a, current_prompt_b, current_prompt_blend, current_neg_prompt, current_strength\n    if resizing:\n        # Configure state before allocating fresh buffers; avoid preparing the\n        # old models only to discard them immediately afterward.\n        for key, name in (('prompt_a', 'current_prompt_a'), ('prompt_b', 'current_prompt_b'),\n                          ('blend', 'current_prompt_blend'), ('neg_prompt', 'current_neg_prompt'),\n                          ('strength', 'current_strength'), ('seed', 'current_seed'), ('mode', 'current_mode')):\n            if key in data:\n                globals()[name] = data[key]\n        output_width, output_height = width, height\n        load_pipeline()\n        return\n    mode = data.get(\"mode\")\n    if mode is not None:\n        new_mode = mode\n        if new_mode != current_mode:\n            current_mode = new_mode\n            logger.info(\"Switching mode to: %s\", current_mode)\n    seed = data.get(\"seed\")\n    if seed is not None and int(seed) != current_seed:\n        current_seed = int(seed)\n        set_seed(current_seed)\n    strength = data.get(\"strength\")\n    if strength is not None and abs(float(strength) - current_strength) > 0.01:\n        apply_strength(strength)\n    apply_prompts_and_blend(\n        data.get(\"prompt_a\", data.get(\"prompt\")), data.get(\"prompt_b\"),\n        data.get(\"blend\"), data.get(\"neg_prompt\"),\n    )\n\ndef compress_frame(arr_bgr, max_bytes=None):\n    \"\"\"\n    Bound payload size without silently changing the selected output resolution.\n    \"\"\"\n    if max_bytes is None:\n        # Give larger outputs a proportional byte budget, retaining the tested\n        # 48KB default for <=512px frames. TLS packets are still only 2KB each.\n        height, width = arr_bgr.shape[:2]\n        max_bytes = max(48000, round(48000 * width * height / (512 * 512)))\n    for q in (60, 50, 40, 30, 20, 10, 5, 1):\n        ret, buf = cv2.imencode('.jpg', arr_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), q])\n        if ret and len(buf) <= max_bytes:\n            return buf.tobytes()\n    raise ValueError(\"Could not encode an output JPEG within the frame byte budget\")\n\n\ndef create_txt2img_pipeline():\n    # StreamDiffusion's bundled TensorRT/Polygraphy path targets old CUDA 11/12\n    # runtimes. Load the native models, then compile their UNets explicitly.\n    return StreamDiffusionWrapper(\n        model_id_or_path=\"stabilityai/sd-turbo\",\n        t_index_list=[0], frame_buffer_size=1,\n        width=output_width, height=output_height, warmup=5,\n        acceleration=\"none\", mode=\"txt2img\",\n        use_lcm_lora=False, use_tiny_vae=True, cfg_type=\"none\",\n        use_denoising_batch=True,\n        device=\"cuda\" if torch.cuda.is_available() else \"cpu\", seed=current_seed,\n    )\n\n\ndef load_pipeline():\n    global stream_txt2img, stream_img2img, pipeline_ready, active_engine, applied_prompt_state\n    global pipeline_revision, img2img_engine\n    if StreamDiffusionWrapper is None:\n        raise RuntimeError(\"StreamDiffusion is not installed. Run the notebook setup cell first.\")\n    pipeline_revision += 1\n    prompt_embedding_cache.clear()\n    applied_prompt_state = None\n    pipeline_ready = False\n    # Release old model buffers before allocating a different resolution.\n    stream_img2img = None\n    stream_txt2img = None\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    active_engine = \"PyTorch (FP16)\"\n    img2img_engine = \"PyTorch (FP16)\"\n    requested = \"TensorRT\" if enable_tensorrt else \"PyTorch FP16\"\n    print(f\">>> Initializing SD-Turbo on {device.upper()}; requested backend: {requested}\", flush=True)\n    print(\">>> Initializing Text-to-Image pipeline (single-image SD-Turbo, t_index_list=[0])...\", flush=True)\n    stream_txt2img = create_txt2img_pipeline()\n    set_seed(current_seed)\n\n    # 2. Image-to-Image / StreamV2V pipeline: initial t_index_list based on current_strength\n    t0_init = max(5, min(44, int(45 - current_strength * 40)))\n    print(f\">>> Initializing Image-to-Image / StreamV2V pipeline (t_index_list=[{t0_init}, 45], strength={current_strength:.2f})...\")\n    try:\n        from streamdiffusion import StreamDiffusion\n        core_pipe = stream_txt2img.stream.pipe\n        stream_core = StreamDiffusion(\n            pipe=core_pipe,\n            t_index_list=[t0_init, 45],\n            torch_dtype=core_pipe.dtype,\n            width=output_width,\n            height=output_height,\n            frame_buffer_size=1,\n            use_denoising_batch=True,\n            cfg_type=\"none\",\n        )\n        if hasattr(stream_txt2img.stream, \"vae\") and stream_txt2img.stream.vae:\n            stream_core.vae = stream_txt2img.stream.vae\n\n        class SharedImg2ImgWrapper:\n            def __init__(self, core):\n                self.stream = core\n            def prepare(self, prompt, **kwargs):\n                self.stream.prepare(prompt, **kwargs)\n            def update_prompt(self, prompt):\n                self.stream.update_prompt(prompt)\n            def update_strength(self, strength: float):\n                # Dynamically maps strength [0.1..1.0] to start timestep index [44..5]\n                t0 = max(5, min(44, int(45 - strength * 40)))\n                new_t_list = [t0, 45]\n                if getattr(self.stream, \"t_list\", None) != new_t_list:\n                    self.stream.t_list = new_t_list\n                    self.stream.prepare(prompt=current_prompt_a, num_inference_steps=50)\n                    logger.info(f\"Updated img2img denoise strength to {strength:.2f} (t_index_list={new_t_list})\")\n            def __call__(self, image):\n                x_output = self.stream(image)\n                pil_out = postprocess_image(x_output, output_type=\"pil\")\n                return pil_out[0] if isinstance(pil_out, list) else pil_out\n\n        stream_img2img = SharedImg2ImgWrapper(stream_core)\n    except Exception as e:\n        print(\">>> Fallback to standalone img2img wrapper:\", e)\n        stream_img2img = StreamDiffusionWrapper(\n            model_id_or_path=\"stabilityai/sd-turbo\",\n            t_index_list=[t0_init, 45],\n            frame_buffer_size=1,\n            width=output_width,\n            height=output_height,\n            warmup=5,\n            acceleration=\"none\",\n            mode=\"img2img\",\n            use_lcm_lora=False,\n            use_tiny_vae=True,\n            cfg_type=\"none\",\n            use_denoising_batch=True,\n            device=device,\n            seed=current_seed,\n        )\n        img2img_engine = \"PyTorch (FP16)\"\n\n    print(\">>> Preparing both pipelines with initial prompt...\")\n    stream_txt2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n    stream_img2img.prepare(prompt=current_prompt_a, num_inference_steps=50)\n\n    if enable_tensorrt:\n        # Batch one serves txt2img; the two-timestep img2img stream uses batch\n        # two. Compile and validate both before reporting readiness.\n        txt_unet = compile_tensorrt_unet(stream_txt2img.stream.unet, output_width, output_height, 1)\n        img_unet = compile_tensorrt_unet(stream_img2img.stream.unet, output_width, output_height, 2)\n        stream_txt2img.stream.unet = txt_unet\n        stream_img2img.stream.unet = img_unet\n        active_engine = img2img_engine = \"TensorRT (UNet)\"\n        print(\">>> TensorRT UNet acceleration enabled for BOTH generation modes; TinyVAE remains PyTorch.\", flush=True)\n\n    print(\">>> Warming up models...\")\n    with torch.no_grad():\n        for _ in range(4):\n            stream_txt2img()\n        dummy_img = Image.new(\"RGB\", (output_width, output_height), color=(128, 128, 128))\n        for _ in range(3):\n            stream_img2img(dummy_img)\n\n    pipeline_ready = True\n    apply_prompts_and_blend()\n    print(\">>> StreamDiffusion pipeline is READY! <<<\", flush=True)\n\n@app.on_event(\"startup\")\nasync def startup_event():\n    loop = asyncio.get_event_loop()\n    await loop.run_in_executor(model_executor, load_pipeline)\n\n@app.get(\"/\")\ndef index():\n    return HTMLResponse(\"\"\"\n    <html>\n        <head><title>StreamDiffusion TouchDesigner Server</title></head>\n        <body style=\"font-family: sans-serif; padding: 40px; background: #121212; color: #fff;\">\n            <h2>StreamDiffusion (SD-Turbo) Colab Server</h2>\n            <p>Status: <span style=\"color: #0f0;\">Running</span></p>\n            <p>Connect your TouchDesigner client via the WebSocket endpoint: <code>/ws/stream</code></p>\n        </body>\n    </html>\n    \"\"\")\n\n\n@app.get(\"/health\")\ndef health():\n    return JSONResponse({\n        \"ready\": pipeline_ready, \"protocol\": PROTOCOL_VERSION,\n        \"mode\": current_mode, \"engine\": engine_for_mode(),\n        \"width\": output_width, \"height\": output_height,\n        \"client_connected\": active_client is not None,\n        \"last_error\": last_pipeline_error,\n        \"requested_engine\": \"TensorRT\" if enable_tensorrt else \"PyTorch (FP16)\",\n    }, status_code=200 if pipeline_ready else 503)\n\n\ndef configuration_error(code):\n    \"\"\"Public status contains no internal exception text or stack information.\"\"\"\n    if code == \"invalid_control\":\n        status = \"Invalid controls: check output preset, mode, seed, strength, and blend values\"\n    elif pipeline_ready:\n        status = \"Configuration failed; previous settings restored. See Colab logs for details\"\n    else:\n        status = \"Configuration failed; see Colab logs, then retry or restart the server\"\n    return {\"type\": \"error\", \"code\": code, \"status\": status,\n            \"ready\": pipeline_ready, \"mode\": current_mode,\n            \"width\": output_width, \"height\": output_height}\n\n@app.websocket(\"/ws/stream\")\nasync def websocket_stream_endpoint(websocket: WebSocket):\n    global active_client\n    # Both modes share one mutable model; an additional controller must not\n    # silently overwrite the active performer's prompts and resolution.\n    if active_client is not None:\n        await websocket.accept()\n        await websocket.send_text(json.dumps({\"type\": \"error\", \"code\": \"server_busy\",\n                                             \"status\": \"Server already has an active TouchDesigner client\"}))\n        await websocket.close(code=1013)\n        return\n    active_client = websocket\n    try:\n        await websocket.accept()\n    except BaseException:\n        active_client = None\n        raise\n    logger.info(\"TouchDesigner client connected to WebSocket.\")\n    loop = asyncio.get_running_loop()\n    send_lock = asyncio.Lock()\n    stop_event = asyncio.Event()\n    frame_available = asyncio.Event()\n    control_available = asyncio.Event()\n    pending_frame = None\n    pending_control = {}\n    assembler = FrameAssembler()\n    frame_counter = 0\n    latest_stats = {}\n\n    async def send_json(data):\n        async with send_lock:\n            await websocket.send_text(json.dumps(data))\n\n    async def send_chunked_frame(jpg_bytes):\n        nonlocal frame_counter\n        frame_counter = (frame_counter + 1) % 256\n        chunk_size = OUTPUT_CHUNK_BYTES\n        total_chunks = (len(jpg_bytes) + chunk_size - 1) // chunk_size\n        if not 1 <= total_chunks <= 255:\n            raise ValueError(\"Output frame exceeds the binary protocol chunk limit\")\n        async with send_lock:\n            for chunk_idx in range(total_chunks):\n                chunk = jpg_bytes[chunk_idx * chunk_size:(chunk_idx + 1) * chunk_size]\n                await websocket.send_bytes(bytes([frame_counter, chunk_idx, total_chunks]) + chunk)\n\n    async def telemetry_loop():\n        while not stop_event.is_set():\n            await asyncio.sleep(1.0)\n            vram, temp = await loop.run_in_executor(None, get_gpu_telemetry)\n            if latest_stats:\n                await send_json({\n                    \"type\": \"stats\", \"engine\": engine_for_mode(),\n                    \"vram\": vram, \"temp\": temp, **latest_stats,\n                })\n\n    async def control_loop():\n        nonlocal pending_control, pending_frame\n        while not stop_event.is_set():\n            await control_available.wait()\n            control_available.clear()\n            data, pending_control = pending_control, {}\n            try:\n                data = validate_controls(data, (output_width, output_height))\n                width, height = requested_resolution(data)\n                resizing = (width, height) != (output_width, output_height) or not pipeline_ready\n                if resizing:\n                    pending_frame = None\n                    frame_available.clear()\n                    assembler.clear()\n                    await send_json({\"type\": \"status\", \"status\": f\"Loading {width} x {height} output...\"})\n                await loop.run_in_executor(model_executor, apply_controls, data)\n                await send_json({\"type\": \"configured\", \"status\": \"Connected (Streaming)\",\n                                 \"width\": output_width, \"height\": output_height, \"engine\": engine_for_mode(),\n                                 \"ready\": pipeline_ready, \"mode\": current_mode})\n                if current_mode != \"img2img\":\n                    pending_frame = None\n                    frame_available.clear()\n                    assembler.clear()\n            except ValueError:\n                logger.exception(\"Invalid control message\")\n                await send_json(configuration_error(\"invalid_control\"))\n            except Exception:\n                logger.exception(\"Error applying control message\")\n                await send_json(configuration_error(\"configuration_failed\"))\n\n    async def generation_loop(mode):\n        nonlocal pending_frame\n        frame_count = 0\n        while not stop_event.is_set():\n            try:\n                frame_bytes = None\n                if mode == \"img2img\":\n                    await frame_available.wait()\n                    frame_available.clear()\n                    frame_bytes, pending_frame = pending_frame, None\n                    if frame_bytes is None:\n                        continue\n                elif not pipeline_ready or current_mode != mode:\n                    await asyncio.sleep(0.01)\n                    continue\n\n                if not pipeline_ready or current_mode != mode:\n                    continue\n                started = time.perf_counter()\n                revision = pipeline_revision\n                result = await loop.run_in_executor(model_executor, process_frame, mode, frame_bytes)\n                # Discard a completed frame if its mode was switched during inference.\n                if result is not None and current_mode == mode and revision == pipeline_revision:\n                    jpg_bytes, inf_ms, process_ms = result\n                    await send_chunked_frame(jpg_bytes)\n                    frame_count += 1\n                    latest_stats.update(inf_ms=round(inf_ms, 1), process_ms=round(process_ms, 1),\n                                        mode=mode, strength=current_strength)\n                    if frame_count % 100 == 1:\n                        logger.info(\"%s: frame #%d (inference %.1fms, processing %.1fms, %d bytes)\",\n                                    mode, frame_count, inf_ms, process_ms, len(jpg_bytes))\n                if mode == \"txt2img\":\n                    await asyncio.sleep(max(0.0, 1.0 / max_stream_fps - (time.perf_counter() - started)))\n            except (WebSocketDisconnect, RuntimeError, OSError):\n                logger.exception(\"Streaming stopped in %s\", mode)\n                stop_event.set()\n                # Wake receive() too, rather than leaving an idle connection alive.\n                await websocket.close()\n                return\n            except Exception:\n                logger.exception(\"Frame processing failed in %s\", mode)\n                await asyncio.sleep(0.05)\n\n    async def supervise(coroutine):\n        try:\n            await coroutine\n        except asyncio.CancelledError:\n            raise\n        except Exception:\n            logger.exception(\"Connection worker failed\")\n            stop_event.set()\n            try:\n                await websocket.close()\n            except Exception:\n                pass\n\n    tasks = []\n    try:\n        vram, temp = await loop.run_in_executor(None, get_gpu_telemetry)\n        await send_json({\n            \"type\": \"welcome\", \"message\": \"StreamDiffusion server connected\",\n            \"engine\": engine_for_mode(), \"prompt_a\": current_prompt_a,\n            \"prompt_b\": current_prompt_b, \"blend\": current_prompt_blend,\n            \"mode\": current_mode, \"seed\": current_seed, \"strength\": current_strength,\n            \"width\": output_width, \"height\": output_height,\n            \"protocol\": PROTOCOL_VERSION, \"ready\": pipeline_ready,\n            \"vram\": vram, \"temp\": temp,\n        })\n        tasks = [asyncio.create_task(supervise(generation_loop(mode))) for mode in (\"txt2img\", \"img2img\")]\n        tasks.append(asyncio.create_task(supervise(telemetry_loop())))\n        tasks.append(asyncio.create_task(supervise(control_loop())))\n        while not stop_event.is_set():\n            message = await websocket.receive()\n            if message.get(\"type\") == \"websocket.disconnect\":\n                break\n            if message.get(\"text\") is not None:\n                try:\n                    data = json.loads(message[\"text\"])\n                    if not isinstance(data, dict):\n                        continue\n                    if data.get(\"type\") == \"ping\":\n                        await send_json({\"type\": \"pong\", \"sent_at\": data.get(\"sent_at\")})\n                    elif data.get(\"type\") == \"control\" or any(\n                        key in data for key in (\"prompt\", \"prompt_a\", \"mode\", \"seed\", \"strength\", \"blend\")\n                    ):\n                        # Merge rapid slider/audio changes while the worker is\n                        # busy; the latest value of each control wins.\n                        pending_control.update(data)\n                        control_available.set()\n                except (ValueError, TypeError):\n                    await send_json({\"type\": \"error\", \"code\": \"invalid_message\",\n                                     \"status\": \"Send a valid JSON control object\"})\n            elif message.get(\"bytes\") is not None and pipeline_ready and current_mode == \"img2img\":\n                raw = message[\"bytes\"]\n                try:\n                    frame_bytes = assembler.feed(raw)\n                except ValueError as exc:\n                    logger.debug(\"Dropped invalid input frame: %s\", exc)\n                    continue\n                if frame_bytes:\n                    # Only one pending input: replace stale watchdog/recovery frames.\n                    # Receiving continues while the worker is busy with inference.\n                    pending_frame = frame_bytes\n                    frame_available.set()\n    except WebSocketDisconnect:\n        logger.info(\"TouchDesigner client disconnected.\")\n    except Exception:\n        logger.exception(\"WebSocket error\")\n    finally:\n        stop_event.set()\n        for task in tasks:\n            task.cancel()\n        await asyncio.gather(*tasks, return_exceptions=True)\n        if active_client is websocket:\n            active_client = None\n\n\nif __name__ == \"__main__\":\n    import uvicorn\n    # JPEG is already compressed; per-message deflate only adds CPU work.\n    uvicorn.run(app, host=\"0.0.0.0\", port=8000, log_level=\"info\", ws_per_message_deflate=False,\n                ws_max_size=MAX_INPUT_FRAME_BYTES + 7)\n"
with open('server.py', 'w', encoding='utf-8') as handle:
    handle.write(server_code)
protocol_code = "\"\"\"Shared presets, control validation, and bounded input frame reassembly.\"\"\"\nimport math\nimport time\n\nPROTOCOL_VERSION = 1\nUPLOAD_CHUNK_BYTES = 4096\nOUTPUT_CHUNK_BYTES = 2048\nMAX_INPUT_FRAME_BYTES = 512 * 1024\nOUTPUT_PRESETS = (\n    ('256', '256 x 256 (1:1 Fast)', 256, 256),\n    ('384', '384 x 384 (1:1)', 384, 384),\n    ('512', '512 x 512 (1:1 Default)', 512, 512),\n    ('768', '768 x 768 (1:1)', 768, 768),\n    ('512x384', '512 x 384 (4:3)', 512, 384),\n    ('768x576', '768 x 576 (4:3)', 768, 576),\n    ('1024x768', '1024 x 768 (4:3 Max)', 1024, 768),\n    ('384x512', '384 x 512 (3:4)', 384, 512),\n    ('576x768', '576 x 768 (3:4)', 576, 768),\n    ('768x1024', '768 x 1024 (3:4 Max)', 768, 1024),\n    ('512x288', '512 x 288 (16:9)', 512, 288),\n    ('768x432', '768 x 432 (16:9)', 768, 432),\n    ('1024x576', '1024 x 576 (16:9)', 1024, 576),\n    ('288x512', '288 x 512 (9:16)', 288, 512),\n    ('432x768', '432 x 768 (9:16)', 432, 768),\n    ('576x1024', '576 x 1024 (9:16)', 576, 1024),\n)\nOUTPUT_RESOLUTIONS = tuple((width, height) for _, _, width, height in OUTPUT_PRESETS)\n\n\ndef integer(value, name):\n    if isinstance(value, bool):\n        raise ValueError(f'{name} must be an integer')\n    try:\n        number = int(value)\n    except (ValueError, TypeError, OverflowError) as exc:\n        raise ValueError(f'{name} must be an integer') from exc\n    if isinstance(value, float) and value != number:\n        raise ValueError(f'{name} must be an integer')\n    return number\n\n\ndef resolution(data, current=(512, 512)):\n    size = (integer(data.get('width', current[0]), 'width'),\n            integer(data.get('height', current[1]), 'height'))\n    if size not in OUTPUT_RESOLUTIONS:\n        raise ValueError('Choose a supported output preset, up to 1024 x 768 or 768 x 1024')\n    return size\n\n\ndef validate_controls(data, current=(512, 512)):\n    \"\"\"Validate the whole patch before mutating any model or application state.\"\"\"\n    if not isinstance(data, dict):\n        raise ValueError('Control message must be a JSON object')\n    clean = {}\n    if 'width' in data or 'height' in data:\n        clean['width'], clean['height'] = resolution(data, current)\n    for name, lower, upper in (('strength', 0.1, 1.0), ('blend', 0.0, 1.0)):\n        if name in data:\n            try:\n                value = float(data[name])\n            except (ValueError, TypeError, OverflowError) as exc:\n                raise ValueError(f'{name} must be a finite number') from exc\n            if isinstance(data[name], bool) or not math.isfinite(value) or not lower <= value <= upper:\n                raise ValueError(f'{name} must be between {lower} and {upper}')\n            clean[name] = value\n    if 'seed' in data:\n        seed = integer(data['seed'], 'seed')\n        if not 0 <= seed <= 4294967295:\n            raise ValueError('seed must be between 0 and 4294967295')\n        clean['seed'] = seed\n    if 'mode' in data:\n        mode = str(data['mode']).lower()\n        aliases = {'txt2img': 'txt2img', 'text': 'txt2img', 'img2img': 'img2img', 'v2v': 'img2img'}\n        if mode not in aliases:\n            raise ValueError('mode must be txt2img or img2img')\n        clean['mode'] = aliases[mode]\n    if 'prompt' in data and 'prompt_a' not in data:\n        data = {**data, 'prompt_a': data['prompt']}\n    for name in ('prompt_a', 'prompt_b', 'neg_prompt'):\n        if name in data:\n            value = data[name]\n            if not isinstance(value, str) or len(value) > 4096:\n                raise ValueError(f'{name} must be text of at most 4096 characters')\n            if name == 'prompt_a' and not value.strip():\n                raise ValueError('Prompt A cannot be empty')\n            clean[name] = value.strip()\n    return clean\n\n\nclass FrameAssembler:\n    \"\"\"Keep one incomplete frame, discarding stale or inconsistent chunks.\"\"\"\n    def __init__(self, max_bytes=MAX_INPUT_FRAME_BYTES, timeout=1.0):\n        self.max_bytes = max_bytes\n        self.timeout = timeout\n        self.completed_id = None\n        self.clear()\n\n    def clear(self):\n        self.frame_id = None\n        self.chunks = []\n        self.size = 0\n        self.started = 0.0\n\n    def feed(self, raw, now=None):\n        now = time.monotonic() if now is None else now\n        if not raw.startswith(b'FRM:'):\n            if not raw.startswith(b'\\xff\\xd8') or len(raw) > self.max_bytes:\n                raise ValueError('Expected a bounded JPEG input frame')\n            self.clear()\n            return raw\n        if len(raw) < 8 or len(raw) > UPLOAD_CHUNK_BYTES + 7:\n            self.clear()\n            raise ValueError('Invalid input chunk size')\n        frame_id, index, count = raw[4:7]\n        if not count or index >= count:\n            self.clear()\n            raise ValueError('Invalid input chunk index or count')\n        if frame_id == self.completed_id:\n            return None\n        if self.frame_id != frame_id or now - self.started > self.timeout:\n            self.clear()\n            self.frame_id = frame_id\n            self.chunks = [None] * count\n            self.started = now\n        if len(self.chunks) != count:\n            self.clear()\n            raise ValueError('Input chunk count changed within a frame')\n        previous = self.chunks[index]\n        self.size += len(raw) - 7 - (len(previous) if previous is not None else 0)\n        if self.size > self.max_bytes:\n            self.clear()\n            raise ValueError('Input frame exceeds the byte limit')\n        self.chunks[index] = raw[7:]\n        if all(chunk is not None for chunk in self.chunks):\n            result = b''.join(self.chunks)\n            self.completed_id = frame_id\n            self.clear()\n            return result\n        return None\n"
with open('bridge_protocol.py', 'w', encoding='utf-8') as handle:
    handle.write(protocol_code)
launcher_code = "\"\"\"Own and monitor the Colab server and tunnel without blocking on log pipes.\"\"\"\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport socket\nimport subprocess\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\n\n\nclass BridgeSession:\n    def __init__(self, tensorrt=False, max_fps=60, root=None, startup_timeout=None, tunnel_timeout=90):\n        if not math.isfinite(float(max_fps)) or not 1 <= float(max_fps) <= 120:\n            raise ValueError('Maximum FPS must be between 1 and 120')\n        self.root = Path(root or Path.cwd()).resolve()\n        self.tensorrt = bool(tensorrt)\n        self.max_fps = float(max_fps)\n        self.startup_timeout = startup_timeout or (1800 if tensorrt else 600)\n        self.tunnel_timeout = tunnel_timeout\n        self.processes = {}\n        self.readers = {}\n        self.writers = {}\n        self.tunnel_url = None\n        self.tunnel_tail = ''\n\n    def _spawn(self, name, command, env=None):\n        logs = self.root / 'bridge_logs'\n        logs.mkdir(exist_ok=True)\n        path = logs / (name + '.log')\n        writer = path.open('w', encoding='utf-8')\n        self.writers[name] = writer\n        # Child processes write to regular files, so a quiet server or a full\n        # unread pipe cannot stall notebook startup or tunnel operation.\n        process = subprocess.Popen(command, cwd=self.root, stdout=writer,\n                                   stderr=subprocess.STDOUT, env=env)\n        self.processes[name] = process\n        self.readers[name] = path.open('r', encoding='utf-8', errors='replace')\n        return process\n\n    def pump_logs(self):\n        for name, reader in self.readers.items():\n            chunk = reader.read(65536)\n            if chunk:\n                print(f'[{name}] {chunk}', end='' if chunk.endswith('\\n') else '\\n', flush=True)\n                if name == 'tunnel':\n                    self.tunnel_tail = (self.tunnel_tail + chunk)[-65536:]\n                    match = re.search(r'https://[a-zA-Z0-9-]+\\.trycloudflare\\.com', self.tunnel_tail)\n                    if match:\n                        self.tunnel_url = match.group(0)\n\n    def _check_processes(self):\n        for name, process in self.processes.items():\n            code = process.poll()\n            if code is not None:\n                raise RuntimeError(f'{name} exited with code {code}. See {self.root / \"bridge_logs\" / (name + \".log\")}')\n\n    def wait_for_ready(self):\n        deadline = time.monotonic() + self.startup_timeout\n        while time.monotonic() < deadline:\n            self.pump_logs()\n            self._check_processes()\n            try:\n                with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as response:\n                    status = json.load(response)\n                    if status.get('ready') is True:\n                        if self.tensorrt and not str(status.get('engine', '')).startswith('TensorRT'):\n                            raise RuntimeError('TensorRT was requested but is not active. Inspect bridge_logs/server.log; no tunnel was started.')\n                        return\n            except (OSError, ValueError, urllib.error.URLError):\n                pass\n            time.sleep(0.2)\n        raise TimeoutError('Server startup timed out. Review bridge_logs/server.log; TensorRT compilation may need more time.')\n\n    def wait_for_tunnel(self):\n        deadline = time.monotonic() + self.tunnel_timeout\n        while time.monotonic() < deadline:\n            self.pump_logs()\n            self._check_processes()\n            if self.tunnel_url:\n                return self.tunnel_url.replace('https://', 'wss://') + '/ws/stream'\n            time.sleep(0.2)\n        raise TimeoutError('Cloudflare did not provide a tunnel URL. Review bridge_logs/tunnel.log.')\n\n    def start(self):\n        self.stop()\n        self.tunnel_url = None\n        self.tunnel_tail = ''\n        try:\n            with socket.create_connection(('127.0.0.1', 8000), timeout=0.5):\n                raise RuntimeError('Port 8000 is already in use. Stop the previous server before launching this session.')\n        except OSError:\n            pass\n        try:\n            if self.tensorrt:\n                from tensorrt_backend import ensure_dependencies\n                ensure_dependencies(self.root)\n            environment = os.environ.copy()\n            environment['MAX_STREAM_FPS'] = str(self.max_fps)\n            environment['ENABLE_TENSORRT'] = '1' if self.tensorrt else '0'\n            self._spawn('server', [sys.executable, '-u', 'server.py'], environment)\n            self.wait_for_ready()\n            self._spawn('tunnel', [str(self.root / 'cloudflared'), 'tunnel', '--url', 'http://127.0.0.1:8000'])\n            return self.wait_for_tunnel()\n        except BaseException:\n            self.stop()\n            raise\n\n    def wait(self):\n        while self.processes:\n            self.pump_logs()\n            self._check_processes()\n            time.sleep(0.2)\n\n    def stop(self):\n        # Only stop children owned by this instance, rather than killing any\n        # unrelated process that happens to hold port 8000.\n        for process in reversed(list(self.processes.values())):\n            if process.poll() is None:\n                try:\n                    process.terminate()\n                    process.wait(timeout=5)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait(timeout=5)\n                except ProcessLookupError:\n                    pass\n        self.processes.clear()\n        for handle in list(self.readers.values()) + list(self.writers.values()):\n            handle.close()\n        self.readers.clear()\n        self.writers.clear()\n"
with open('colab_launcher.py', 'w', encoding='utf-8') as handle:
    handle.write(launcher_code)
tensorrt_code = "\"\"\"Torch-TensorRT acceleration for current Colab Python/CUDA runtimes.\"\"\"\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\n\ndef package_requirement(torch_version, python_version=None):\n    python_version = python_version or sys.version_info[:2]\n    if tuple(python_version) < (3, 11):\n        raise RuntimeError('TensorRT acceleration needs Python 3.11 or newer. Use a current Colab runtime or disable TensorRT.')\n    match = re.match(r'^(\\d+)\\.(\\d+)\\.', torch_version)\n    if match is None or tuple(map(int, match.groups())) < (2, 11):\n        raise RuntimeError('This TensorRT backend requires PyTorch 2.11 or newer. Disable TensorRT on older runtimes.')\n    return f'torch-tensorrt=={match[1]}.{match[2]}.*'\n\n\ndef ensure_dependencies(root):\n    \"\"\"Install matching wheels and verify imports; never silently request FP16.\"\"\"\n    import torch\n    requirement = package_requirement(torch.__version__)\n    if not torch.cuda.is_available() or not torch.version.cuda:\n        raise RuntimeError('TensorRT requires a CUDA GPU runtime.')\n    probe = [sys.executable, '-c', 'import torch_tensorrt, tensorrt; print(\"Torch-TensorRT\", torch_tensorrt.__version__, \"TensorRT\", tensorrt.__version__)']\n    result = subprocess.run(probe, cwd=root, capture_output=True, text=True)\n    if result.returncode == 0:\n        print(result.stdout.strip(), flush=True)\n        return\n    logs = Path(root) / 'bridge_logs'\n    logs.mkdir(exist_ok=True)\n    log_path = logs / 'tensorrt-setup.log'\n    cuda_tag = 'cu' + torch.version.cuda.replace('.', '')\n    command = [sys.executable, '-m', 'pip', 'install', requirement, f'torch=={torch.__version__}',\n               '--extra-index-url', f'https://download.pytorch.org/whl/{cuda_tag}']\n    print(f'Installing {requirement} for PyTorch {torch.__version__} / CUDA {torch.version.cuda}.', flush=True)\n    print(f'Installation log: {log_path}', flush=True)\n    with log_path.open('w', encoding='utf-8') as log:\n        installed = subprocess.run(command, cwd=root, stdout=log, stderr=subprocess.STDOUT, timeout=900)\n    if installed.returncode:\n        print(log_path.read_text(encoding='utf-8', errors='replace')[-10000:], flush=True)\n        raise RuntimeError('TensorRT installation failed. See bridge_logs/tensorrt-setup.log; uncheck TensorRT to use PyTorch.')\n    result = subprocess.run(probe, cwd=root, capture_output=True, text=True)\n    if result.returncode:\n        print(result.stderr, flush=True)\n        raise RuntimeError('TensorRT imports failed after installation. See the startup output; acceleration was not enabled.')\n    print(result.stdout.strip(), flush=True)\n\n\ndef cache_directory(root, width, height, batch_size, torch_version, trt_version, capability):\n    tag = re.sub(r'[^A-Za-z0-9._-]', '_', f'torch-{torch_version}-trt-{trt_version}-sm{capability[0]}{capability[1]}')\n    return Path(root) / tag / f'{width}x{height}' / f'batch-{batch_size}'\n\n\nclass TensorRTUNet:\n    \"\"\"Adapt the compiled tensor output to StreamDiffusion's UNet call contract.\"\"\"\n    backend = 'TensorRT (UNet)'\n\n    def __init__(self, compiled, source, batch_size):\n        self.compiled = compiled\n        self.config = source.config\n        self.dtype = source.dtype\n        self.device = source.device\n        self.batch_size = batch_size\n\n    def __call__(self, sample, timestep, encoder_hidden_states, **kwargs):\n        import torch\n        if sample.shape[0] != self.batch_size:\n            raise ValueError('TensorRT UNet batch size does not match the compiled mode')\n        result = self.compiled(sample, timestep.to(dtype=torch.float32), encoder_hidden_states)\n        return (result,)\n\n    def to(self, *args, **kwargs):\n        return self\n\n    def eval(self):\n        return self\n\n\ndef compile_unet(unet, width, height, batch_size, cache_root='engines/torch-tensorrt'):\n    import torch\n    import torch_tensorrt\n    package_requirement(torch.__version__)\n    if not torch.cuda.is_available():\n        raise RuntimeError('TensorRT compilation requires a CUDA GPU.')\n\n    class UNetGraph(torch.nn.Module):\n        def __init__(self, model):\n            super().__init__()\n            self.model = model\n\n        def forward(self, sample, timestep, encoder_hidden_states):\n            return self.model(sample, timestep, encoder_hidden_states=encoder_hidden_states, return_dict=False)[0]\n\n    graph = UNetGraph(unet).eval()\n    inputs = [\n        torch.randn((batch_size, unet.config.in_channels, height // 8, width // 8), device=unet.device, dtype=unet.dtype),\n        torch.full((batch_size,), 999.0, device=unet.device, dtype=torch.float32),\n        torch.randn((batch_size, 77, unet.config.cross_attention_dim), device=unet.device, dtype=unet.dtype),\n    ]\n    cache = cache_directory(cache_root, width, height, batch_size, torch.__version__,\n                            torch_tensorrt.__version__, torch.cuda.get_device_capability())\n    cache.mkdir(parents=True, exist_ok=True)\n    print(f'>>> Compiling TensorRT UNet: {width} x {height}, batch {batch_size}. First use can take several minutes.', flush=True)\n    with torch.no_grad():\n        expected = graph(*inputs).detach()\n        # Export selects CUDA FlashAttention for cross-attention, whose unequal\n        # query/key sequence lengths lack a Torch-TensorRT converter. Export\n        # those layers as equivalent matmul/softmax operations instead. Keep\n        # self-attention's efficient processor and compare against the original.\n        from diffusers.models.attention_processor import AttnProcessor\n        unet.set_attn_processor({\n            name: AttnProcessor() if name.endswith('attn2.processor') else processor\n            for name, processor in unet.attn_processors.items()\n        })\n        compiled = torch_tensorrt.compile(\n            # Current Torch-TensorRT uses explicit typing. The UNet weights\n            # and latents already carry FP16; timesteps remain FP32.\n            graph, ir='dynamo', inputs=inputs, use_explicit_typing=True,\n            # Decompose Diffusers' baddbmm into supported primitive operations.\n            enable_experimental_decompositions=True,\n            require_full_compilation=True, workspace_size=2 * 1024**3,\n            cache_built_engines=True, reuse_cached_engines=True, engine_cache_dir=str(cache),\n        )\n        engine_count = sum(type(module).__name__ in ('TorchTensorRTModule', 'PythonTorchTensorRTModule')\n                           for module in compiled.modules())\n        if engine_count == 0:\n            raise RuntimeError('Compilation produced no TensorRT engines; acceleration was not enabled.')\n        actual = compiled(*inputs)\n        torch.testing.assert_close(actual, expected, rtol=0.05, atol=0.05)\n        torch.cuda.synchronize()\n    print(f'>>> TensorRT UNet batch {batch_size} compiled and output validation passed ({engine_count} engine(s)).', flush=True)\n    return TensorRTUNet(compiled, unet, batch_size)\n"
with open('tensorrt_backend.py', 'w', encoding='utf-8') as handle:
    handle.write(tensorrt_code)
print('Bridge server and helpers written successfully.')


In [ ]:
# @title Step 4: Launch Server and Cloudflare Tunnel
# @markdown TensorRT may take several minutes to compile engines for a new output size.
ENABLE_TENSORRT = False # @param {type:"boolean"}
MAX_FPS = 60 # @param {type:"integer"}

from html import escape
from IPython.display import display, HTML
if 'bridge_session' in globals():
    bridge_session.stop()
import importlib
import colab_launcher
importlib.reload(colab_launcher)
bridge_session = colab_launcher.BridgeSession(tensorrt=ENABLE_TENSORRT, max_fps=MAX_FPS)
try:
    ws_url = bridge_session.start()
    display(HTML(f'<h3>StreamDiffusion is ready</h3><p>Paste this URL into TouchDesigner:</p><pre>{escape(ws_url)}</pre>'))
    bridge_session.wait()
except KeyboardInterrupt:
    print('Stopping this bridge session...')
finally:
    bridge_session.stop()
